# "Ruse of Reuse" Hackathon Starter Kit ( Embedding + Passim)


## ▶️ Google Colab

**This section is intended to be run in a Google Colab environment. If you are running this notebook locally, skip the cells below and go to 'Evaluation Pipeline' section.**

Run this cell to configure environment, install all necessary packages, and  download the data.

If you are running this notebook locally, follow the instructions in the `README.md` to set up your environment and load the data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glsch/solemne-data-atelier/blob/main/notebooks/method_evaluation_two_methods.ipynb)






In [ ]:
%%capture
from google.colab import userdata
import os

try:
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    os.environ['GITHUB_TOKEN'] = ''
    print("Warning: GITHUB_TOKEN secret not found. Defaulting to empty string.")

try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except userdata.SecretNotFoundError:
    os.environ['OPENAI_API_KEY'] = ''
    print("Warning: OPENAI_API_KEY secret not found. Defaulting to empty string.")

!gh auth setup-git
!gh repo clone glsch/solemne-data-atelier


In [ ]:
%%capture

!cp -a ./solemne-data-atelier/. ./
!rm -rf ./solemne-data-atelier


In [ ]:
%%capture
!python -m pip install -e .


In [ ]:
!python -m solemne_data_atelier download


Although the task data is delivered already pre-processed, you can produce it from raw data by running:


In [ ]:
!python -m solemne_data_atelier preprocess


To run the Passim baseline (optional), passim must be installed. For details how to install passim on any machine, see [this manual](https://programminghistorian.org/en/lessons/detecting-text-reuse-with-passim) and [](https://github.com/dasmiq/passim/blob/main/docs/passim_quickstart.ipynb) To install in Colab, simply run

In [ ]:
!pip install git+https://github.com/dasmiq/passim.git

## Evaluation Pipeline

1. Make sure to have the following file structure:
```
solemne_data_atelier/
├── data/                                # Should be downloaded
│   ├── raw/
│   ├── task/
│   ├── vectorstores/
│   ├── book_mapping.tsv
│   ├── reference_mapping.json
│   └── bible.tsv
├── notebooks/
│   ├── method_evaluation_two_methods.ipynb
│   └── method_evaluation.ipynb
└── src/                                 # Package code
    ├── solemne_data_atelier/
    └── ...
```
2. Run cells top-to-bottom once to load helper functions and the data.
3. To get baseline performance, run the rest of the notebook without modifying the code `participant_method()`  it to get .
3. To implement your own approach, modify the `participant_method()`. **It is important that the structure of output remains the same!**

For more information, see docstrings of the

### Expected Output Structure
Metric computation depends on `participant_method()` returning correctly structured output.

`participant_method()` must return: `List[Dict[str, Any]]`

Each dictionary represents **one predicted pair**. For example,
```
{
    "problem_id": "D.LEO-0458-541",
    "reference: "john_1:3"
}
```
`"reference"` key must contain only **one** biblical verse structured as follows:
``
<book_code>_<chapter>:<verse>
``
Book codes must be consistent with those used in `bible.tsv`. They can be found either in `bible.tsv` or, in a more concise form, in `book_mapping.tsv`.

More formally:
* `problem_id` (required, `str`)
  - Must be the filename stems from `data/task/problems/*.txt`.
  > Example: `"D.LEO-0458-541"`
* `reference` (required, `str`)
  - A single biblical verse reference in format: `<book_code>_<chapter>:<verse>`.
  **Span references, e.g., ``john_1:3-5`` must be expanded into three separate verses each represented by their own dictionary!**
  - Book code should be consistent with `bible.tsv`. Lowercase is applied by the evaluator.
  - Example: `"john_1:3"`

Dictionary may include any number of optional keys. For example, the proposed baseline method also outputs contains:
*`score` (optional, `float`)
  - Cosine similarity. Not used directly in scoring, but useful for debugging and analysis.
*`method` (optional, `str`)
  - Method name label, `"simple_embedding"`.

### Important Evaluation Behavior

- Scoring uses only `(problem_id, reference)` pairs.
- Duplicate predictions of the same pair are deduplicated automatically.
- Missing required keys (`problem_id`, `reference`) effectively drop a row from scoring.
- Extra keys are allowed but ignored by the scorer.

### Examples
#### Valid output
```python
[
  {"problem_id": "D.LEO-0458-541", "reference": "john_1:3"},
  {"problem_id": "D.LEO-0458-541", "reference": "john_1:4", "score": 0.81, "method": "baseline"},
]
```

#### Invalid output
```
# Problem identifier does not correspond to a filename stem from data/task/problems
[
  {
    "problem_id": "123",
    "reference": "john_1:3"
  },
  ...
]

# "reference" key is missing
[
  {
    "problem_id": "D.LEO-0458-541",
  },
  ...
]

# Reference to multiple verses
[
  {
    "problem_id": "D.LEO-0458-541",
    "reference": "john_1:3-5"
  },
  ...
]

# Invalid book code
[
  {
    "problem_id": "D.LEO-0458-541",
    "reference": "jhn_1:3"
  },
  ...
]
```


In [1]:
from __future__ import annotations

import copy
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

from solemne_data_atelier import setup_logging
from solemne_data_atelier.logging import ModuleLoggingConfig

setup_logging(
    ModuleLoggingConfig(
        default_level=logging.INFO,
        log_file="method_evaluation.log",
    )
)

logger = logging.getLogger("method_evaluation_dual")
logger.setLevel(logging.INFO)


In [2]:
from solemne_data_atelier.evaluation import (
    find_project_root,
    load_bible_tsv,
    load_task_ground_truth,
    load_task_problems,
)

PROJECT_ROOT = find_project_root()
TASK_DIR = PROJECT_ROOT / "data" / "task"
CHROMA_DIR = PROJECT_ROOT / "data" / "vectorstores" / "chroma"
PASSIM_RUNS_DIR = PROJECT_ROOT / "passim_runs"
SIMPLE_EMBEDDING_RUNS_DIR = PROJECT_ROOT / "simple_embedding_runs"

problems = load_task_problems(TASK_DIR)
ground_truth = load_task_ground_truth(TASK_DIR)
bible_df = load_bible_tsv(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded problems: {len(problems)}")
print(f"Loaded ground-truth docs: {len(ground_truth)}")
print(f"Bible verses: {len(bible_df)}")


Project root: /Users/glebschmidt/PycharmProjects/solemne-data-atelier
Loaded problems: 53
Loaded ground-truth docs: 53
Bible verses: 35809


/Users/glebschmidt/PycharmProjects/solemne-data-atelier/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Evaluation API

- `run_method_on_dataset()` is method-agnostic and should be reused for any future approach.
- `score_predictions()` computes precision/recall/F1 over a list of `(problem_id, reference)` pairs provided as dictionaries. Automatically deduplicated during scoring is applied in case of multiple identical problem-verse pairs.


In [3]:
from solemne_data_atelier.evaluation import MethodFn, run_method_on_dataset, score_predictions

### Methods: Simple Embedding and Passim

This notebook exposes a unified evaluation flow for two interchangeable methods:

- `simple_embedding`
- `passim`

Switch methods in the experiment cell by changing `METHOD_NAME`.


In [4]:
from solemne_data_atelier.methods.passim import (
    build_passim_method_context,
    passim_method,
    save_passim_metrics,
)
from solemne_data_atelier.methods.simple_embedding import (
    build_embedding_method_context,
    save_simple_embedding_run,
    simple_embedding_method,
)


def build_method_context(
    *,
    method_name: str,
    problems_by_id: Dict[str, str],
    ground_truth_by_problem: Optional[Dict[str, List[str]]] = None,
    bible_df: Optional[pd.DataFrame] = None,
    project_root: Optional[Path] = None,
    chroma_dir: Optional[Path] = None,
    passim_runs_dir: Optional[Path] = None,
    max_problems: Optional[int] = None,
    method_kwargs: Optional[Dict[str, Any]] = None,
    **kwargs,
) -> Dict[str, Any]:
    method = str(method_name).strip().lower()
    selected_problem_ids = sorted(problems_by_id.keys())
    if max_problems is not None:
        selected_problem_ids = selected_problem_ids[: int(max_problems)]

    if method == "simple_embedding":
        if chroma_dir is None:
            raise ValueError("chroma_dir is required for simple_embedding")

        ctx = build_embedding_method_context(
            chroma_dir=Path(chroma_dir),
            **kwargs,
        )
        ctx["method_name"] = "simple_embedding"
        ctx["selected_problem_ids"] = selected_problem_ids
        ctx["method_kwargs"] = dict(method_kwargs or {})
        return ctx

    if method == "passim":
        if bible_df is None:
            raise ValueError("bible_df is required for passim")
        if ground_truth_by_problem is None:
            raise ValueError("ground_truth_by_problem is required for passim")
        if project_root is None:
            raise ValueError("project_root is required for passim")
        if passim_runs_dir is None:
            raise ValueError("passim_runs_dir is required for passim")

        return build_passim_method_context(
            problems_by_id=problems_by_id,
            ground_truth_by_problem=ground_truth_by_problem,
            bible_df=bible_df,
            project_root=Path(project_root),
            passim_runs_dir=Path(passim_runs_dir),
            problem_ids=selected_problem_ids,
            **kwargs,
        )

    raise ValueError("method_name must be one of: simple_embedding, passim")


In [5]:
def participant_method(problem_id: str, problem_text: str, method_context: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Unified participant method API for both implementations.

    Expected output format per item:
    {
      "problem_id": <str>,
      "reference": <str>,
      "score": <float, optional>,
      "method": <str, optional>
    }
    """
    method_name = str(method_context.get("method_name", "")).strip().lower()

    if method_name == "simple_embedding":
        return simple_embedding_method(
            problem_id,
            problem_text,
            method_context,
            **dict(method_context.get("method_kwargs", {})),
        )

    if method_name == "passim":
        return passim_method(problem_id, problem_text, method_context)

    raise ValueError(f"Unsupported method_name in context: {method_name}")


### Running Experiments

- Start with `max_problems=5` for quick iteration, then evaluate all problems.
- If model/collection mismatch occurs, check the mapping file in `data/vectorstores/chroma`.
- Keep the same API contract so your method can be swapped into the evaluator without changes.


In [7]:
METHOD_NAME = "simple_embedding"
MAX_PROBLEMS = None

if METHOD_NAME == "simple_embedding":
    method_context = build_method_context(
        method_name="simple_embedding",
        problems_by_id=problems,
        chroma_dir=CHROMA_DIR,
        max_problems=MAX_PROBLEMS,
        provider="hf",
        model_name="bowphs/LaBerta",
        device="mps",
        method_kwargs={
            "mode": "char",
            "char_chunk_size": 100,
            "char_chunk_overlap": 50,
            "top_k": 2,
            "similarity_threshold": 0.75,
        },
    )

elif METHOD_NAME == "passim":
    method_context = build_method_context(
        method_name="passim",
        problems_by_id=problems,
        ground_truth_by_problem=ground_truth,
        bible_df=bible_df,
        project_root=PROJECT_ROOT,
        passim_runs_dir=PASSIM_RUNS_DIR,
        max_problems=MAX_PROBLEMS,
        chunking_enabled=True,
        mode="char",
        char_chunk_size=100,
        char_chunk_overlap=50,
        min_chunk_chars=25,
        passim_options={
            "docwise": True,
            "fields": ["group"],
            "filterpairs": "group = 'bible' AND group2 = 'task'",
            "output-format": "json",
            "n": 8,
            "min-match": 3,
            "min-align": 20,
            "minDF": 1,
            "maxDF": 750,
            "g": 350,
            "beam": 1000,
            "dst-overlap": 0.5,
            "src-overlap": 0.70,
            "pcopy": 0.75,
            "log-level": "WARN",
        },
    )

else:
    raise ValueError("METHOD_NAME must be 'simple_embedding' or 'passim'.")

predictions = run_method_on_dataset(
    participant_method,
    problems,
    method_context,
    problem_ids=method_context.get("selected_problem_ids"),
    show_progress=True,
)

subset_ground_truth = {
    pid: ground_truth.get(pid, [])
    for pid in method_context.get("selected_problem_ids", sorted(problems.keys()))
}

metrics = score_predictions(predictions, subset_ground_truth)
print(pd.Series(metrics))
print(f"Predictions produced: {len(predictions)}")

result_to_push = copy.deepcopy(metrics)


if METHOD_NAME == "simple_embedding":
    simple_run_dir = save_simple_embedding_run(
        simple_embedding_runs_dir=SIMPLE_EMBEDDING_RUNS_DIR,
        method_context=method_context,
        method_kwargs=dict(method_context.get("method_kwargs", {})),
        metrics=metrics,
        predictions_count=len(predictions),
        selected_problem_ids=method_context.get("selected_problem_ids", sorted(problems.keys())),
    )
    print(f"Simple embedding run folder: {simple_run_dir}")

if METHOD_NAME == "passim":
    passim_metrics_path = save_passim_metrics(
        run_dir=method_context["run_dir"],
        metrics=metrics,
        predictions_count=len(predictions),
    )
    print(f"Passim metrics YAML: {passim_metrics_path}")
    print(f"Passim run folder: {method_context['run_dir']}")
    print(f"Passim config: {method_context['config_path']}")
    print(f"Passim visual HTML: {method_context['reshaped_dir'] / 'visual_matches.html'}")


2026-03-17 12:26:10,900 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: bowphs/LaBerta
2026-03-17 12:26:11,220 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-03-17 12:26:11,222 - sentence_transformers.SentenceTransformer - WARNING - No sentence-transformers model found with name bowphs/LaBerta. Creating a new one with mean pooling.
2026-03-17 12:26:11,338 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-17 12:26:11,446 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 12:26:11,447 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-17 12:26:11,459 

true_positives     419.000000
false_positives    122.000000
false_negatives    434.000000
precision            0.774492
recall               0.491208
f1                   0.601148
dtype: float64
Predictions produced: 541
Simple embedding run folder: /Users/glebschmidt/PycharmProjects/solemne-data-atelier/simple_embedding_runs/20260317_122643
